# Convergenza: Fitness classica vs Extended Fitness

Per ogni iterazione salvo la variazione relativa massima di $F$ e $Q$ e i valori minimi di $F$ e $Q$ (normalizzati a media unitaria), per la Fitness classica e per l'Extended Fitness con filtro degli import ($a=1$, `tensor_window = 3`), anno 2002, HS4 e HS6.

Le due iterazioni sono riscritte qui per poter tracciare i minimi passo per passo; l'ultima cella verifica che l'Extended Fitness tracciata coincida con `compute_extended_fitness_with_imports`.

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import sys
from pathlib import Path

ROOT = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src.algorithms import compute_alpha
from src.algorithms_with_imports import compute_extended_fitness_with_imports

DATA = ROOT / "data" / "HS02_all_years"
OUT = ROOT / "results" / "convergenza"
OUT.mkdir(parents=True, exist_ok=True)

N_ITER = 300
TENSOR_WINDOW = 3
YEAR = 2002

In [2]:
def load(res, year=YEAR):
    base = DATA / f"{res}_digits" / "alligned" / "binarized"
    Mcp_export = pd.read_parquet(base / "Mcp_export" / f"Mcp_export_alligned_Y{year}.parquet")
    Mcp_import = pd.read_parquet(base / "Mcp_import" / f"Mcp_import_alligned_Y{year}.parquet")
    C = pd.read_parquet(base / "C" / f"C_{res}_digits_HS02.parquet")
    return Mcp_export, Mcp_import, C


def rel_change(new, old):
    return np.max(np.abs(new - old) / np.maximum(new, 1e-300))


def track_fitness(Mcp, n_iter=N_ITER):
    """Fitness classica. Si escludono paesi e prodotti senza export (righe/colonne
    nulle), che renderebbero indefinito l'aggiornamento."""
    M = Mcp.values.astype(float)
    M = M[M.sum(axis=1) > 0][:, M.sum(axis=0) > 0]
    F, Q = np.ones(M.shape[0]), np.ones(M.shape[1])
    rows = []
    for i in range(n_iter):
        F_old, Q_old = F, Q
        F = M @ Q_old
        with np.errstate(divide="ignore"):
            Q = 1.0 / (M.T @ (1.0 / F_old))
        F, Q = F / F.mean(), Q / Q.mean()
        rows.append((i + 1, max(rel_change(F, F_old), rel_change(Q, Q_old)), F.min(), Q.min()))
    return pd.DataFrame(rows, columns=["iter", "dmax", "minF", "minQ"])


def track_extended_fitness(Mcp_export, Mcp_import, C, alpha, K=TENSOR_WINDOW, n_iter=N_ITER):
    """Stessa iterazione di compute_extended_fitness_with_imports (a cui si passa C.T),
    con il tracciamento dei minimi. Restituisce anche F e Q finali per il controllo."""
    M_E = Mcp_export.values.astype(float)
    M_I = Mcp_import.values.astype(float)
    Ct = C.T.values.astype(float)
    B = sp.csr_matrix(np.ascontiguousarray(Ct.T))
    n_c, n_p = M_E.shape
    F, Q = np.ones(n_c), np.ones(n_p)
    Delta = np.zeros((K + 1, n_c, n_p))
    rows = []
    for i in range(n_iter):
        F_old, Q_old = F.copy(), Q.copy()
        R = Delta[K] * M_I
        for m in range(K - 1, 0, -1):
            R = Delta[m] * M_I + alpha * (R @ B)
        S = alpha * np.asarray(R @ B)
        F = M_E @ Q_old - (S * M_E).sum(axis=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            Delta_new = 1.0 / (M_E.T @ (1.0 / F_old))
        Delta_new[~np.isfinite(Delta_new)] = 0.0
        Delta_new /= Delta_new.mean()
        Q = alpha * (Ct @ Q_old) + Delta_new
        mean_Q = Q.mean()
        Q, F = Q / mean_Q, F / F.mean()
        Delta[1:] = Delta[:-1] / mean_Q
        Delta[0] = Delta_new / mean_Q
        rows.append((i + 1, max(rel_change(F, F_old), rel_change(Q, Q_old)), F.min(), Q.min()))
    return pd.DataFrame(rows, columns=["iter", "dmax", "minF", "minQ"]), F, Q

In [3]:
checks = {}
for res in (4, 6):
    Mcp_export, Mcp_import, C = load(res)
    alpha = compute_alpha(C)

    classica = track_fitness(Mcp_export).assign(algoritmo="fitness")
    estesa, F_end, Q_end = track_extended_fitness(Mcp_export, Mcp_import, C, alpha)
    estesa = estesa.assign(algoritmo="extended_fitness")

    table = pd.concat([classica, estesa], ignore_index=True)
    table.to_parquet(OUT / f"convergenza_{res}_digits.parquet")
    checks[res] = (Mcp_export, Mcp_import, C, alpha, F_end, Q_end)

    for name, h in table.groupby("algoritmo"):
        conv = h.loc[h.dmax < 1e-6, "iter"]
        print(f"HS{res} {name:17s} iter a 1e-6: {conv.min() if len(conv) else '-':>4}  "
              f"min F finale: {h.minF.iloc[-1]:.1e}  min Q finale: {h.minQ.iloc[-1]:.1e}")

HS4 extended_fitness  iter a 1e-6:   26  min F finale: 6.1e-03  min Q finale: 2.8e-02
HS4 fitness           iter a 1e-6:    -  min F finale: 2.8e-16  min Q finale: 1.9e-15
HS6 extended_fitness  iter a 1e-6:   24  min F finale: 4.0e-03  min Q finale: 1.0e-02
HS6 fitness           iter a 1e-6:    -  min F finale: 3.3e-52  min Q finale: 3.1e-51


In [4]:
# Controllo: l'iterazione tracciata coincide con la funzione in src
for res, (Mcp_export, Mcp_import, C, alpha, F_end, Q_end) in checks.items():
    Q_ref, F_ref, *_ = compute_extended_fitness_with_imports(
        Mcp_export, Mcp_import, C.T, alpha=alpha, tensor_window=TENSOR_WINDOW,
        max_iter=N_ITER, tol=1e-14, track_convergence=False)
    print(f"HS{res}: max |dF| = {np.max(np.abs(F_end - F_ref.values)):.1e}, "
          f"max |dQ| = {np.max(np.abs(Q_end - Q_ref.values)):.1e}")

HS4: max |dF| = 1.8e-15, max |dQ| = 4.1e-15
HS6: max |dF| = 5.8e-15, max |dQ| = 2.5e-14
